# Service 1 — Étape 5 : export du modèle fine-tuné en GGUF (4 bits)

**But :** transformer « modèle de base + adaptateur LoRA » en **un seul fichier `.gguf`** qu'Ollama sait faire tourner
sur le processeur du PC (étape 6).

1. **Fusion** : l'adaptateur est ajouté définitivement aux poids du modèle (plus de calcul séparé).
2. **Conversion** au format GGUF (format de llama.cpp, utilisé par Ollama).
3. **Quantification Q4_K_M** : les poids sont stockés sur environ 4 bits → environ 4,7 Go au lieu d'environ 15 Go.

**Avant de lancer :** GPU T4, puis *Exécution → Tout exécuter*. Durée : 20 à 40 min.
Les fichiers intermédiaires (environ 30 Go) restent sur le disque de Colab ; **seul le `.gguf` final va sur le Drive**.

## 1. Drive, code et chemins

In [1]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/cardioagent-ops"
REPO_URL = "https://github.com/yasssmine2003/CardioAgent-Ops.git"
REPO_DIR = "/content/CardioAgent-Ops"
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}/llm_finetuning

ADAPTER_DIR = f"{DRIVE}/llm/adapter_qwen2.5-7b-cardio"   # résultat de l'étape 3
EXPORT_DIR = "/content/gguf_export"                     # disque de Colab (fichiers temporaires)
GGUF_DRIVE_DIR = f"{DRIVE}/llm/gguf"                    # destination finale
GGUF_NAME = "qwen2.5-7b-cardio-Q4_K_M.gguf"
QUANT = "q4_k_m"                                        # [C] meilleur compromis taille / qualité courant
os.makedirs(GGUF_DRIVE_DIR, exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = f"file:{DRIVE}/mlruns"
!df -h /content | tail -1

Mounted at /content/drive
/content/CardioAgent-Ops/llm_finetuning
overlay         113G   93M  113G   1% /


## 2. Installation

In [2]:
!pip install -q unsloth "mlflow>=2.17,<3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires pyarrow>=23.0.1, but you have pyarrow 19.0.1 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


## 3. Chargement du modèle fine-tuné (base + adaptateur)

In [3]:
import torch
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(ADAPTER_DIR, max_seq_length=2048, load_in_4bit=True)
print(f"Mémoire GPU : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1121 09:46:06.751000 637 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1121 09:46:06.796000 637 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.11.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Mémoire GPU : 5.7 Go


## 4. Fusion + conversion + quantification
Unsloth fait les 3 opérations : il fusionne l'adaptateur dans les poids (en 16 bits), installe llama.cpp,
convertit en GGUF puis quantifie en Q4_K_M. Les messages de compilation de llama.cpp sont normaux.

In [4]:
import time
start = time.perf_counter()
model.save_pretrained_gguf(EXPORT_DIR, tokenizer, quantization_method=QUANT)
export_minutes = (time.perf_counter() - start) / 60
print(f"Export terminé en {export_minutes:.0f} min")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/gguf_export/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (14 outputs left out)



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [04:01<00:00, 60.43s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [03:28<00:00, 52.10s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_export`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12588-mix-c0c3de8 (app-b12588-mix-c0c3de8-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/gguf_export_gguf/Qwen2.5-7B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed

## 5. Vérification et copie sur le Drive

In [5]:
import glob, hashlib, shutil

candidates = [p for p in glob.glob("/content/**/*.gguf", recursive=True) if QUANT.upper() in p.upper()]
print("Fichiers GGUF trouvés :", candidates)
src = max(candidates, key=os.path.getmtime)        # le plus récent = celui qu'on vient de créer
size_gb = os.path.getsize(src) / 1e9

sha = hashlib.sha256()                              # empreinte : prouve que le fichier n'a pas changé
with open(src, "rb") as f:
    for block in iter(lambda: f.read(1 << 24), b""):
        sha.update(block)
sha256 = sha.hexdigest()

dst = f"{GGUF_DRIVE_DIR}/{GGUF_NAME}"
shutil.copy(src, dst)
print(f"GGUF : {dst}\nTaille : {size_gb:.2f} Go\nSHA-256 : {sha256}")

Fichiers GGUF trouvés : ['/content/gguf_export_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf']
GGUF : /content/drive/MyDrive/cardioagent-ops/llm/gguf/qwen2.5-7b-cardio-Q4_K_M.gguf
Taille : 4.68 Go
SHA-256 : 50340991cd33a600afc30a32cacab9f9b3a541ad7f4dd07e6072a0865eddac22


## 6. Enregistrement dans MLflow
Le fichier (environ 4,7 Go) n'est pas copié dans MLflow : on enregistre son chemin, sa taille et son empreinte.

In [6]:
import mlflow
mlflow.set_experiment("cardioagent-llm")
with mlflow.start_run(run_name="export_gguf"):
    mlflow.set_tags({"stage": "export", "family": "deployment", "gguf_sha256": sha256})
    mlflow.log_params({"base_model": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", "adapter": ADAPTER_DIR,
                       "quantization": QUANT, "gguf_path": dst, "merged_adapter": True})
    mlflow.log_metrics({"gguf_size_gb": size_gb, "export_minutes": export_minutes})
print("Run MLflow : export_gguf")

Run MLflow : export_gguf
